# Conservation of Momentum

For any collision or explosion, the **total momentum is conserved** — as long as no external force acts:

$$m_1u_1 + m_2u_2 = m_1v_1 + m_2v_2$$

- $m_1, m_2$ — the masses of the two blocks (kg)
- $u_1, u_2$ — their velocities **before** (m/s)
- $v_1, v_2$ — their velocities **after** (m/s)

Velocity to the **right is positive**, to the **left is negative**.

## The three experiments

1. **Collision** — the blocks collide and **bounce apart**.
2. **Collision and stick** — the blocks collide and **stick together**, moving as one afterwards.
3. **Explosion** — the blocks start **together at rest**; a spring pushes them apart, so the total momentum stays zero.

Set the masses and initial velocities with the sliders, then press **Run**.


In [1]:
import numpy as np

import matplotlib.pyplot as plt

from matplotlib.patches import Rectangle, FancyArrowPatch

from matplotlib.animation import FuncAnimation

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    %pip install -q ipywidgets
    import ipywidgets as widgets

from IPython.display import display, HTML





# ------------------------------------------------------------

# The physics

# ------------------------------------------------------------



TRACK_LEN = 16.0   # length of track shown (m)

TRACK_Y = 1.0      # height of the track line

BLOCK_H = 1.4      # block height on the diagram

ARROW_Y = TRACK_Y + BLOCK_H + 0.7   # height of the velocity arrows

WALL = 0.2         # blocks stop this far from each end of the track





def block_width(m):

    """Heavier block = wider block (display only)."""

    return 1.2 + 0.35 * m





def elastic_velocities(m1, m2, u1, u2):

    """Velocities after a perfectly elastic collision (bounce)."""

    v1 = ((m1 - m2) * u1 + 2 * m2 * u2) / (m1 + m2)

    v2 = ((m2 - m1) * u2 + 2 * m1 * u1) / (m1 + m2)

    return v1, v2





def stick_velocity(m1, m2, u1, u2):

    """Common velocity after the blocks stick together."""

    return (m1 * u1 + m2 * u2) / (m1 + m2)





def explosion_velocities(m1, m2, E):

    """Blocks start together at rest; energy E pushes them apart.

    Momentum: 0 = m1*v1 + m2*v2

    Energy:   E = 0.5*m1*v1**2 + 0.5*m2*v2**2

    Block 1 moves left (v1 < 0), block 2 moves right (v2 > 0).

    """

    v2 = np.sqrt(2 * E * m1 / (m2 * (m1 + m2)))

    v1 = -m2 * v2 / m1

    return v1, v2





# ------------------------------------------------------------

# Drawing

# ------------------------------------------------------------



def build_track_figure(m1, m2, title):

    """Draw the track and the two blocks; return artists the animation moves."""

    fig, ax = plt.subplots(figsize=(9, 3.0))

    ax.set_xlim(-0.5, TRACK_LEN + 0.5)

    ax.set_ylim(-0.4, 4.5)



    # the track

    ax.axhline(TRACK_Y, color="black", linewidth=2)

    for x in np.arange(0, TRACK_LEN + 0.01, 1.0):

        ax.plot([x, x], [TRACK_Y - 0.18, TRACK_Y], color="black", linewidth=0.8)



    w1, w2 = block_width(m1), block_width(m2)



    block1 = Rectangle((0, TRACK_Y), w1, BLOCK_H, facecolor="lightblue",

                       edgecolor="black", linewidth=2)

    block2 = Rectangle((0, TRACK_Y), w2, BLOCK_H, facecolor="lightsalmon",

                       edgecolor="black", linewidth=2)

    ax.add_patch(block1)

    ax.add_patch(block2)



    mass1_text = ax.text(0, 0, f"m₁\n{m1:g} kg", ha="center", va="center", fontsize=11)

    mass2_text = ax.text(0, 0, f"m₂\n{m2:g} kg", ha="center", va="center", fontsize=11)



    arrow1 = FancyArrowPatch((0, 0), (0, 0), arrowstyle="-|>",

                             mutation_scale=18, linewidth=2, color="black")

    arrow2 = FancyArrowPatch((0, 0), (0, 0), arrowstyle="-|>",

                             mutation_scale=18, linewidth=2, color="black")

    ax.add_patch(arrow1)

    ax.add_patch(arrow2)



    vel1_text = ax.text(0, 0, "", ha="center", va="bottom", fontsize=11)

    vel2_text = ax.text(0, 0, "", ha="center", va="bottom", fontsize=11)



    flash = ax.plot([], [], marker="*", markersize=30, linestyle="",

                    color="gold", markeredgecolor="darkorange",

                    markeredgewidth=1.5)[0]



    ax.set_title(title, fontsize=13)

    ax.set_xlabel("Position along track (m)")

    ax.set_xticks(np.arange(0, TRACK_LEN + 1, 2))

    ax.set_yticks([])

    for side in ("left", "right", "top"):

        ax.spines[side].set_visible(False)

    fig.tight_layout()



    artists = dict(

        block1=block1, block2=block2,

        mass1_text=mass1_text, mass2_text=mass2_text,

        arrow1=arrow1, arrow2=arrow2,

        vel1_text=vel1_text, vel2_text=vel2_text,

        flash=flash,

    )

    return fig, artists





def place(artists, m1, m2, x1, x2, s1, s2, lab1, lab2, flash_xy=None):

    """Move every artist: x = left edge of each block, s = velocity to show."""

    w1, w2 = block_width(m1), block_width(m2)



    artists["block1"].set_x(x1)

    artists["block2"].set_x(x2)

    artists["mass1_text"].set_position((x1 + w1 / 2, TRACK_Y + BLOCK_H / 2))

    artists["mass2_text"].set_position((x2 + w2 / 2, TRACK_Y + BLOCK_H / 2))



    c1 = x1 + w1 / 2

    c2 = x2 + w2 / 2

    L1 = 0.45 * max(-6.5, min(6.5, s1))   # cap arrow length (display only)

    L2 = 0.45 * max(-6.5, min(6.5, s2))

    L1 = max(-(c1 - 0.1), min(TRACK_LEN - 0.1 - c1, L1))  # and keep it on the track

    L2 = max(-(c2 - 0.1), min(TRACK_LEN - 0.1 - c2, L2))

    artists["arrow1"].set_positions((c1, ARROW_Y), (c1 + L1, ARROW_Y))

    artists["arrow2"].set_positions((c2, ARROW_Y), (c2 + L2, ARROW_Y))

    artists["arrow1"].set_visible(abs(s1) > 1e-9)

    artists["arrow2"].set_visible(abs(s2) > 1e-9)

    artists["vel1_text"].set_position((c1 + 0.5 * L1, ARROW_Y + 0.15))

    artists["vel2_text"].set_position((c2 + 0.5 * L2, ARROW_Y + 0.15))

    artists["vel1_text"].set_text(lab1)

    artists["vel2_text"].set_text(lab2)



    if flash_xy is None:

        artists["flash"].set_data([], [])

    else:

        artists["flash"].set_data([flash_xy[0]], [flash_xy[1]])





def show_track(m1, m2, x1, x2, s1, s2, lab1, lab2, title):

    """Draw one still picture of the track (used before Run is pressed)."""

    fig, artists = build_track_figure(m1, m2, title)

    place(artists, m1, m2, x1, x2, s1, s2, lab1, lab2)

    display(fig)

    plt.close(fig)





# ------------------------------------------------------------

# Building the animations

# ------------------------------------------------------------



def vel_label(name, v):

    return f"{name} = {v:.1f} m/s" if abs(v) > 1e-9 else "at rest"





def collision_frames(m1, m2, u1, u2, mode):

    """Frame list for a collision. mode is "elastic" or "stick".

    Returns (frames, collided). Each frame is

    (x1, x2, s1, s2, label1, label2, flash_xy).

    """

    w1, w2 = block_width(m1), block_width(m2)

    x10, x20 = 2.0, 10.0



    # Do they collide while still on the track?

    approach = u1 - u2

    t_c = xc1 = None

    on_track = False

    if approach > 1e-12:

        t_c = ((x20 - x10) - w1) / approach

        xc1 = x10 + u1 * t_c

        on_track = (xc1 >= WALL) and (xc1 + w1 + w2 <= TRACK_LEN - WALL)



    frames = []

    lab_u1, lab_u2 = vel_label("u₁", u1), vel_label("u₂", u2)



    if on_track:

        if mode == "elastic":

            v1, v2 = elastic_velocities(m1, m2, u1, u2)

        else:

            v1 = v2 = stick_velocity(m1, m2, u1, u2)

        xc2 = xc1 + w1



        # how long to run after the collision: until the first block hits an end

        if abs(v1) < 1e-9 and abs(v2) < 1e-9:

            t_after = 1.2

        else:

            t_after = 3.0

            for v, xc, w in ((v1, xc1, w1), (v2, xc2, w2)):

                if v > 1e-9:

                    t_after = min(t_after, max(0.4, (TRACK_LEN - WALL - w - xc) / v))

                elif v < -1e-9:

                    t_after = min(t_after, max(0.4, (xc - WALL) / (-v)))



        lab_v1, lab_v2 = vel_label("v₁", v1), vel_label("v₂", v2)

        flash_until = t_c + 0.3 * t_after



        times = list(np.linspace(0, t_c, 18))

        times += list(np.linspace(t_c, t_c + t_after, 26))[1:]



        for t in times:

            if t < t_c:

                frames.append((x10 + u1 * t, x20 + u2 * t,

                               u1, u2, lab_u1, lab_u2, None))

            else:

                dt = t - t_c

                x1 = xc1 + v1 * dt

                x2 = xc2 + v2 * dt

                if mode == "stick":

                    if v1 > 0 and x2 > TRACK_LEN - WALL - w2:

                        x2 = TRACK_LEN - WALL - w2

                        x1 = x2 - w1

                    elif v1 < 0 and x1 < WALL:

                        x1 = WALL

                        x2 = x1 + w1

                else:

                    x1 = min(max(x1, WALL), TRACK_LEN - WALL - w1)

                    x2 = min(max(x2, WALL), TRACK_LEN - WALL - w2)

                flash = (xc1 + w1, TRACK_Y + BLOCK_H / 2) if t <= flash_until else None

                frames.append((x1, x2, v1, v2, lab_v1, lab_v2, flash))



        return frames, True



    # ---- no collision on the track: the blocks just slide (and stop at the ends)

    t_end = 3.0

    for u, x0, w in ((u1, x10, w1), (u2, x20, w2)):

        if u > 1e-9:

            t_end = min(t_end, (TRACK_LEN - WALL - w - x0) / u)

        elif u < -1e-9:

            t_end = min(t_end, (x0 - WALL) / (-u))

    t_end = max(t_end, 0.4)



    for t in np.linspace(0, t_end, 30):

        x1 = min(max(x10 + u1 * t, WALL), TRACK_LEN - WALL - w1)

        x2 = min(max(x20 + u2 * t, WALL), TRACK_LEN - WALL - w2)

        frames.append((x1, x2, u1, u2, lab_u1, lab_u2, None))



    return frames, False





def explosion_frames(m1, m2, E):

    """Frame list for an explosion: blocks start together at rest."""

    w1, w2 = block_width(m1), block_width(m2)

    v1, v2 = explosion_velocities(m1, m2, E)   # v1 < 0 (left), v2 > 0 (right)



    # start touching at the centre of the track

    x1_0 = TRACK_LEN / 2 - w1

    x2_0 = TRACK_LEN / 2

    flash_xy = (TRACK_LEN / 2, TRACK_Y + BLOCK_H / 2)



    t_end = 3.0

    t_end = min(t_end, max(0.4, (x1_0 - WALL) / (-v1)))

    t_end = min(t_end, max(0.4, (TRACK_LEN - WALL - w2 - x2_0) / v2))



    lab_v1, lab_v2 = vel_label("v₁", v1), vel_label("v₂", v2)



    frames = []

    for _ in range(8):   # hold: sitting together at rest

        frames.append((x1_0, x2_0, 0.0, 0.0, "at rest", "at rest", None))

    for t in np.linspace(0, t_end, 28):

        x1 = max(x1_0 + v1 * t, WALL)

        x2 = min(x2_0 + v2 * t, TRACK_LEN - WALL - w2)

        flash = flash_xy if t <= 0.35 * t_end else None

        frames.append((x1, x2, v1, v2, lab_v1, lab_v2, flash))

    return frames





def momentum_animation(m1, m2, frames, title, interval=70):

    """Turn a frame list into an in-browser animation."""

    fig, artists = build_track_figure(m1, m2, title)



    def update(i):

        x1, x2, s1, s2, lab1, lab2, flash_xy = frames[i]

        place(artists, m1, m2, x1, x2, s1, s2, lab1, lab2, flash_xy)

        return tuple(artists.values())



    anim = FuncAnimation(fig, update, frames=len(frames), interval=interval, repeat=False)

    plt.close(fig)

    return HTML(anim.to_jshtml())





# ------------------------------------------------------------

# The momentum sums shown beside each experiment

# ------------------------------------------------------------



def summary_collision(m1, m2, u1, u2, v1, v2, stuck, collided):

    p0 = m1 * u1 + m2 * u2

    before = (f"<b>Before:</b> m₁u₁ + m₂u₂ = ({m1:g})({u1:g}) + ({m2:g})({u2:g}) "

              f"= <b>{p0:.2f} kg m/s</b><br>")

    if not collided:

        return (before +

                "<span style='color:firebrick'><b>No collision on the track!</b> "

                "Make u₁ bigger than u₂ so block 1 catches block 2.</span>")

    if stuck:

        p1 = (m1 + m2) * v1

        ke0 = 0.5 * m1 * u1**2 + 0.5 * m2 * u2**2

        ke1 = 0.5 * (m1 + m2) * v1**2

        return (before +

                f"<b>After:</b> (m₁ + m₂)v = ({m1:g} + {m2:g})({v1:.2f}) "

                f"= <b>{p1:.2f} kg m/s</b> ✓<br>"

                f"Stuck together: <b>v = {v1:.2f} m/s</b><br>"

                f"<span style='color:dimgray'>KE before = {ke0:.2f} J, KE after = {ke1:.2f} J "

                f"— {ke0 - ke1:.2f} J lost as heat, sound, deformation</span>")

    p1 = m1 * v1 + m2 * v2

    ke0 = 0.5 * m1 * u1**2 + 0.5 * m2 * u2**2

    ke1 = 0.5 * m1 * v1**2 + 0.5 * m2 * v2**2

    return (before +

            f"<b>After:</b> m₁v₁ + m₂v₂ = ({m1:g})({v1:.2f}) + ({m2:g})({v2:.2f}) "

            f"= <b>{p1:.2f} kg m/s</b> ✓<br>"

            f"<b>v₁ = {v1:.2f} m/s</b>, <b>v₂ = {v2:.2f} m/s</b><br>"

            f"<span style='color:dimgray'>KE before = {ke0:.2f} J, KE after = {ke1:.2f} J "

            f"— elastic collision, no KE lost</span>")





def summary_explosion(m1, m2, E, v1, v2):

    p1 = m1 * v1 + m2 * v2

    ke1 = 0.5 * m1 * v1**2 + 0.5 * m2 * v2**2

    return ("<b>Before:</b> both at rest — total momentum = <b>0 kg m/s</b><br>"

            f"<b>After:</b> m₁v₁ + m₂v₂ = ({m1:g})({v1:.2f}) + ({m2:g})({v2:.2f}) "

            f"= <b>{p1:.2f} kg m/s</b> ✓<br>"

            f"<b>v₁ = {v1:.2f} m/s</b> (left), <b>v₂ = {v2:.2f} m/s</b> (right)<br>"

            f"<span style='color:dimgray'>Explosion energy = {E:g} J "

            f"→ KE of the blocks = {ke1:.2f} J</span>")


## 1. Collision — the blocks bounce apart

A spring buffer makes the collision **elastic**: the blocks bounce off each other.

Momentum is conserved:

$$m_1u_1 + m_2u_2 = m_1v_1 + m_2v_2$$

and because a perfectly elastic collision also conserves kinetic energy, the final velocities are

$$v_1 = \frac{(m_1-m_2)u_1 + 2m_2u_2}{m_1+m_2}, \qquad v_2 = \frac{(m_2-m_1)u_2 + 2m_1u_1}{m_1+m_2}$$

**Try:** equal masses with $u_2 = 0$ — the first block stops dead and passes on *all* of its momentum. Then try a heavy block hitting a light one, and a light block hitting a heavy one.


In [2]:
# ============================================================

# DEMO 1: elastic collision -- the blocks bounce apart

# ============================================================



m1_slider1 = widgets.FloatSlider(

    value=2.0, min=0.5, max=5.0, step=0.5,

    description="m₁ (kg):", continuous_update=False

)

m2_slider1 = widgets.FloatSlider(

    value=1.0, min=0.5, max=5.0, step=0.5,

    description="m₂ (kg):", continuous_update=False

)

u1_slider1 = widgets.FloatSlider(

    value=3.0, min=-4.0, max=4.0, step=0.5,

    description="u₁ (m/s):", continuous_update=False

)

u2_slider1 = widgets.FloatSlider(

    value=0.0, min=-4.0, max=4.0, step=0.5,

    description="u₂ (m/s):", continuous_update=False

)



run_button1 = widgets.Button(description="Run", button_style="primary", icon="play")

reset_button1 = widgets.Button(description="Reset")

value_label1 = widgets.HTML()

sim_output1 = widgets.Output(layout=widgets.Layout(width="720px"))





def values1():

    return (m1_slider1.value, m2_slider1.value,

            u1_slider1.value, u2_slider1.value)





def show_values1():

    m1, m2, u1, u2 = values1()

    v1, v2 = elastic_velocities(m1, m2, u1, u2)

    _, collided = collision_frames(m1, m2, u1, u2, mode="elastic")

    value_label1.value = summary_collision(m1, m2, u1, u2, v1, v2,

                                           stuck=False, collided=collided)





def show_initial1():

    m1, m2, u1, u2 = values1()

    show_values1()

    show_track(m1, m2, 2.0, 10.0, u1, u2,

               vel_label("u₁", u1), vel_label("u₂", u2),

               "Collision — press Run")





def run_clicked1(button):

    m1, m2, u1, u2 = values1()

    show_values1()

    frames, collided = collision_frames(m1, m2, u1, u2, mode="elastic")

    if collided:

        title = "Elastic collision — they bounce apart"

    else:

        title = "No collision on the track"

    with sim_output1:

        sim_output1.clear_output(wait=True)

        display(momentum_animation(m1, m2, frames, title))





def reset_clicked1(button):

    with sim_output1:

        sim_output1.clear_output(wait=True)

        show_initial1()





run_button1.on_click(run_clicked1)

reset_button1.on_click(reset_clicked1)



controls1 = widgets.VBox([

    widgets.HTML("<h3>Controls</h3>"),

    m1_slider1,

    m2_slider1,

    u1_slider1,

    u2_slider1,

    widgets.HBox([run_button1, reset_button1]),

    value_label1

], layout=widgets.Layout(width="330px"))



display(widgets.HBox([controls1, sim_output1],

                     layout=widgets.Layout(align_items="flex-start", gap="18px")))



with sim_output1:

    show_initial1()


## 2. Collision and stick

Velcro pads (or Plasticine) make the blocks **stick together** and move as one with a common velocity $v$.

Momentum is still conserved:

$$m_1u_1 + m_2u_2 = (m_1 + m_2)\,v \qquad\Rightarrow\qquad v = \frac{m_1u_1 + m_2u_2}{m_1+m_2}$$

Kinetic energy is **not** conserved here — some becomes heat, sound and deformation of the velcro.

**Try:** equal masses with equal and opposite velocities ($u_1 = -u_2$). The total momentum is zero, so the stuck blocks stop dead.


In [3]:
# ============================================================

# DEMO 2: the blocks collide and stick together

# ============================================================



m1_slider2 = widgets.FloatSlider(

    value=2.0, min=0.5, max=5.0, step=0.5,

    description="m₁ (kg):", continuous_update=False

)

m2_slider2 = widgets.FloatSlider(

    value=1.0, min=0.5, max=5.0, step=0.5,

    description="m₂ (kg):", continuous_update=False

)

u1_slider2 = widgets.FloatSlider(

    value=3.0, min=-4.0, max=4.0, step=0.5,

    description="u₁ (m/s):", continuous_update=False

)

u2_slider2 = widgets.FloatSlider(

    value=-1.0, min=-4.0, max=4.0, step=0.5,

    description="u₂ (m/s):", continuous_update=False

)



run_button2 = widgets.Button(description="Run", button_style="primary", icon="play")

reset_button2 = widgets.Button(description="Reset")

value_label2 = widgets.HTML()

sim_output2 = widgets.Output(layout=widgets.Layout(width="720px"))





def values2():

    return (m1_slider2.value, m2_slider2.value,

            u1_slider2.value, u2_slider2.value)





def show_values2():

    m1, m2, u1, u2 = values2()

    v = stick_velocity(m1, m2, u1, u2)

    _, collided = collision_frames(m1, m2, u1, u2, mode="stick")

    value_label2.value = summary_collision(m1, m2, u1, u2, v, v,

                                           stuck=True, collided=collided)





def show_initial2():

    m1, m2, u1, u2 = values2()

    show_values2()

    show_track(m1, m2, 2.0, 10.0, u1, u2,

               vel_label("u₁", u1), vel_label("u₂", u2),

               "Collision and stick — press Run")





def run_clicked2(button):

    m1, m2, u1, u2 = values2()

    show_values2()

    frames, collided = collision_frames(m1, m2, u1, u2, mode="stick")

    if collided:

        title = "They collide and stick together"

    else:

        title = "No collision on the track"

    with sim_output2:

        sim_output2.clear_output(wait=True)

        display(momentum_animation(m1, m2, frames, title))





def reset_clicked2(button):

    with sim_output2:

        sim_output2.clear_output(wait=True)

        show_initial2()





run_button2.on_click(run_clicked2)

reset_button2.on_click(reset_clicked2)



controls2 = widgets.VBox([

    widgets.HTML("<h3>Controls</h3>"),

    m1_slider2,

    m2_slider2,

    u1_slider2,

    u2_slider2,

    widgets.HBox([run_button2, reset_button2]),

    value_label2

], layout=widgets.Layout(width="330px"))



display(widgets.HBox([controls2, sim_output2],

                     layout=widgets.Layout(align_items="flex-start", gap="18px")))



with sim_output2:

    show_initial2()


## 3. Explosion

The blocks start **together at rest**, so the total momentum is zero. A spring-loaded plunger (or small explosion) pushes them apart.

Because momentum is conserved:

$$0 = m_1v_1 + m_2v_2 \qquad\Rightarrow\qquad m_1v_1 = -m_2v_2$$

The **lighter block shoots off faster**, in the opposite direction — this is also how a rocket works.

The slider sets the energy released by the spring. It all becomes kinetic energy of the blocks:

$$E = \tfrac{1}{2}m_1v_1^2 + \tfrac{1}{2}m_2v_2^2$$

**Try:** make $m_2$ much smaller than $m_1$ — the small block flies off fast while the big one barely recoils.


In [ ]:
# ============================================================

# DEMO 3: explosion -- the blocks start together at rest

# ============================================================



m1_slider3 = widgets.FloatSlider(

    value=2.0, min=0.5, max=5.0, step=0.5,

    description="m₁ (kg):", continuous_update=False

)

m2_slider3 = widgets.FloatSlider(

    value=1.0, min=0.5, max=5.0, step=0.5,

    description="m₂ (kg):", continuous_update=False

)

energy_slider3 = widgets.FloatSlider(

    value=20.0, min=5.0, max=100.0, step=5.0,

    description="Energy (J):", continuous_update=False

)



run_button3 = widgets.Button(description="Run", button_style="primary", icon="play")

reset_button3 = widgets.Button(description="Reset")

value_label3 = widgets.HTML()

sim_output3 = widgets.Output(layout=widgets.Layout(width="720px"))





def values3():

    return m1_slider3.value, m2_slider3.value, energy_slider3.value





def show_values3():

    m1, m2, E = values3()

    v1, v2 = explosion_velocities(m1, m2, E)

    value_label3.value = summary_explosion(m1, m2, E, v1, v2)





def show_initial3():

    m1, m2, E = values3()

    show_values3()

    w1 = block_width(m1)

    show_track(m1, m2, TRACK_LEN / 2 - w1, TRACK_LEN / 2, 0.0, 0.0,

               "at rest", "at rest", "Explosion — press Run")





def run_clicked3(button):

    m1, m2, E = values3()

    show_values3()

    frames = explosion_frames(m1, m2, E)

    with sim_output3:

        sim_output3.clear_output(wait=True)

        display(momentum_animation(m1, m2, frames,

                                   "Explosion — momentum stays zero"))





def reset_clicked3(button):

    with sim_output3:

        sim_output3.clear_output(wait=True)

        show_initial3()





run_button3.on_click(run_clicked3)

reset_button3.on_click(reset_clicked3)



controls3 = widgets.VBox([

    widgets.HTML("<h3>Controls</h3>"),

    m1_slider3,

    m2_slider3,

    energy_slider3,

    widgets.HBox([run_button3, reset_button3]),

    value_label3

], layout=widgets.Layout(width="330px"))



display(widgets.HBox([controls3, sim_output3],

                     layout=widgets.Layout(align_items="flex-start", gap="18px")))



with sim_output3:

    show_initial3()
